# LeetCode 357 — Count Numbers with Unique Digits bằng Python

Link đề: https://leetcode.com/problems/count-numbers-with-unique-digits/ · Độ khó: **Medium** · Chủ đề: Math (toán tổ hợp), Dynamic Programming, Backtracking (quay lui), Digit DP, Bitmask

Skeleton LeetCode:

```python
class Solution:
    def countNumbersWithUniqueDigits(self, n: int) -> int:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu năm cách làm với vét cạn và với đếm hoán vị bằng `itertools.permutations` cho mọi `n` từ `0` đến `8`). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho số nguyên `n`, đếm các số nguyên `x` với `0 <= x < 10^n` mà **các chữ số của `x` đều khác nhau** (ví dụ `121`, `100`, `11` không thỏa; `123`, `907`, `5` thỏa).
2. Nắm các công cụ Python: `str(x)`, `len(set(s))` để kiểm tra trùng chữ số, `range(10**n)`, **quy tắc nhân** (đếm hoán vị/chỉnh hợp) với `math.perm`, **bitmask** (`mask >> j & 1`, `mask | 1 << j`), `@functools.cache` (ghi nhớ kết quả đệ quy), và đệ quy/quay lui.
3. Đi từ cách **duyệt mọi số** → **đếm theo độ dài bằng quy tắc nhân** (`9 · 9 · 8 · ...`) → **quay lui đếm từng số** → **Digit DP** `dfs(i, mask, lead)` → **công thức `math.perm` / bảng đáp án**.
4. Hiểu mấu chốt: đếm theo **số chữ số `k`** (`k = 1..n`): chữ số đầu có `9` cách (`1..9`, không phải `0`), chữ số tiếp theo có `9` cách (`0..9` bỏ chữ số đã dùng), rồi `8`, `7`, ...; cộng thêm số `0`. Vì chỉ có `10` chữ số nên với `k > 10` không còn số nào.
5. Tránh các lỗi kinh điển: quên **`n = 0` ⇒ `1`** (số `0`), cho chữ số đầu `10` cách (tính cả `0` đứng đầu), **chỉ đếm số có đúng `n` chữ số** mà quên các số ngắn hơn, bỏ cờ `lead` trong Digit DP, và duyệt vét cạn `10^8` số khi `n = 8`.

## 2. Đề bài & ví dụ

Cho số nguyên `n`, trả về **số lượng** các số nguyên `x` thỏa `0 <= x < 10^n` mà **mọi chữ số của `x` đều khác nhau** (số có chữ số trùng nhau bị loại).

**Ví dụ 1**

- Input: `n = 2`
- Output: `91`
- Giải thích: đếm các số trong khoảng `0 <= x < 100`, **loại** `11, 22, 33, 44, 55, 66, 77, 88, 99` (`100 − 9 = 91`).

**Ví dụ 2**

- Input: `n = 0`
- Output: `1`
- Giải thích: khoảng `0 <= x < 1` chỉ có số `0`.

**Ví dụ tự dựng (đối chiếu lại đề gốc nếu cần):** `n = 1` → `10` (các số `0..9`); `n = 3` → `739`.

**Ràng buộc:** `0 <= n <= 8`.

In [1]:
import time
from functools import cache
from itertools import permutations
from math import perm

examples = [
    (2, 91),
    (0, 1),
    (1, 10),      # tự dựng
    (3, 739),     # tự dựng
]
for n, expected in examples:
    print("n =", n, "-> kỳ vọng", expected)

n = 2 -> kỳ vọng 91
n = 0 -> kỳ vọng 1
n = 1 -> kỳ vọng 10
n = 3 -> kỳ vọng 739


## 3. Kiến thức nền

### 3.1. Kiểm tra chữ số khác nhau: `str` + `set`

Một số có chữ số đôi một khác nhau ⇔ **số chữ số = số chữ số phân biệt**. Đổi số thành chuỗi `s = str(x)`, rồi `set(s)` gom các ký tự phân biệt: `len(set(s)) == len(s)`.

In [2]:
def has_unique_digits(x):
    s = str(x)
    return len(set(s)) == len(s)

for x in [0, 5, 10, 11, 123, 121, 907, 100, 98765]:
    print(x, str(x), set(str(x)), "->", has_unique_digits(x))

0 0 {'0'} -> True
5 5 {'5'} -> True
10 10 {'0', '1'} -> True
11 11 {'1'} -> False
123 123 {'2', '3', '1'} -> True
121 121 {'2', '1'} -> False
907 907 {'9', '7', '0'} -> True
100 100 {'0', '1'} -> False
98765 98765 {'6', '8', '9', '7', '5'} -> True


### 3.2. `range(10 ** n)`

`10 ** n` là luỹ thừa: `10**0 = 1`, `10**2 = 100`, `10**8 = 100000000`. `range(10 ** n)` sinh các số `0, 1, ..., 10^n − 1`, đúng khoảng `0 <= x < 10^n` của đề. Lưu ý: khi `n = 8` thì có **một trăm triệu** số.

In [3]:
print(10 ** 0, 10 ** 2, 10 ** 8)
print(list(range(10 ** 0)), len(range(10 ** 2)), len(range(10 ** 8)))

1 100 100000000
[0] 100 100000000


### 3.3. Quy tắc nhân và chỉnh hợp

Nếu chọn lần lượt: bước 1 có `a` cách, bước 2 có `b` cách (không phụ thuộc cách chọn ở bước 1), ... thì tổng số cách là `a · b · ...` (**quy tắc nhân**).

Số cách xếp `k` chữ số **đôi một khác nhau** từ `10` chữ số, có tính thứ tự: `10 · 9 · 8 ⋯ (10 − k + 1)`. Đây là **chỉnh hợp** `P(10, k)`; Python có `math.perm(10, k)`. Nhưng số **không được bắt đầu bằng `0`**, nên chữ số đầu chỉ có `9` cách (`1..9`), các chữ số sau lần lượt `9, 8, 7, ...` cách (có thể là `0` nhưng phải khác các chữ số đã dùng).

In [4]:
print(perm(10, 3), 10 * 9 * 8)        # 720 720: xếp 3 chữ số khác nhau, cho phép 0 đứng đầu
print(9 * 9 * 8)                       # 648: số 3 chữ số, chữ số đầu 1..9
# kiểm chứng bằng vét cạn nhỏ: bao nhiêu số 3 chữ số (100..999) có chữ số khác nhau?
print(sum(1 for x in range(100, 1000) if has_unique_digits(x)))     # 648

720 720
648
648


### 3.4. Bitmask: nhớ "chữ số nào đã dùng" bằng một số nguyên

Có đúng `10` chữ số `0..9`, nên dùng `mask` (số nguyên) mà **bit thứ `j`** bằng `1` nghĩa là "chữ số `j` đã dùng". Các thao tác:

- kiểm tra: `mask >> j & 1` (dịch phải `j` bit rồi lấy bit cuối) — `1` nếu đã dùng;
- đánh dấu dùng: `mask | (1 << j)` (bật bit `j`);
- `1 << j` là số có duy nhất bit `j` bằng `1` (`2^j`).

In [5]:
mask = 0
for d in [3, 7, 3]:
    if mask >> d & 1:
        print("chữ số", d, "đã dùng")
    else:
        mask = mask | (1 << d)
        print("dùng chữ số", d, "-> mask =", bin(mask))
print(bin(mask), [j for j in range(10) if mask >> j & 1])       # [3, 7]

dùng chữ số 3 -> mask = 0b1000
dùng chữ số 7 -> mask = 0b10001000
chữ số 3 đã dùng
0b10001000 [3, 7]


### 3.5. Đệ quy và `functools.cache`

Hàm đệ quy `f(args)` có thể bị gọi **lặp** với cùng `args`. `@cache` (từ `functools`) **ghi nhớ** kết quả theo `args` (các tham số phải băm được, ví dụ số nguyên, bool, tuple) để lần sau trả ngay — đó là **ghi nhớ (memoization)**, một dạng quy hoạch động (DP) "từ trên xuống".

In [6]:
calls = 0

@cache
def fib(k):
    global calls
    calls += 1
    return k if k < 2 else fib(k - 1) + fib(k - 2)

print(fib(30), "số lần thực sự tính:", calls)       # 31 lần, thay vì ~2.7 triệu lần không cache

832040 số lần thực sự tính: 31


### 3.6. Sinh hoán vị với `itertools.permutations`

`permutations(range(10), k)` sinh mọi bộ `k` chữ số **đôi một khác nhau** có thứ tự. Ta sẽ dùng nó làm **oracle độc lập**: đếm các bộ có phần tử đầu khác `0` (số có đúng `k` chữ số).

In [7]:
print(list(permutations([1, 2, 3], 2)))
print(sum(1 for p in permutations(range(10), 2) if p[0] != 0))      # 81: số có đúng 2 chữ số, khác nhau

[(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
81


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Chia các số `x` trong `[0, 10^n)` theo **số chữ số** `k` (với `k = 1..n`), cộng riêng số `0`.

**`n = 1`:** `0..9`: mười số, đều chỉ có một chữ số ⇒ `10`.

**`n = 2`:** các số `0..9` (`10` số) và các số hai chữ số `10..99` (`90` số). Trong `90` số này, các số có hai chữ số **giống nhau** là `11, 22, ..., 99` (`9` số). Vậy `90 − 9 = 81` số hai chữ số hợp lệ. Tổng: `10 + 81 = 91`. ✔.

Cách nghĩ theo "xếp chữ số": chữ số hàng chục: `1..9` ⇒ `9` cách; chữ số hàng đơn vị: `0..9` trừ chữ số đã dùng ⇒ `9` cách; `9 · 9 = 81`. ✔.

**`n = 3`:** thêm số có `3` chữ số: hàng trăm `9` cách, hàng chục `9` cách (`0..9` trừ hàng trăm), hàng đơn vị `8` cách (`0..9` trừ hai chữ số đã dùng): `9 · 9 · 8 = 648`. Tổng: `91 + 648 = 739`. ✔.

| độ dài `k` | cách chọn từng chữ số | số lượng | cộng dồn (gồm số `0`) |
|---|---|---|---|
| `0` (chỉ số `0`) | — | `1` | `1` |
| `1` | `9` (chữ số `1..9`) + số `0` đã tính | `9` | `10` |
| `2` | `9 · 9` | `81` | `91` |
| `3` | `9 · 9 · 8` | `648` | `739` |
| `4` | `9 · 9 · 8 · 7` | `4536` | `5275` |

In [8]:
total = 1                       # số 0
ways = 9                        # chữ số đầu: 1..9
for k in range(1, 5):
    if k > 1:
        ways *= 11 - k          # chữ số thứ k: còn 10 - (k - 1) = 11 - k lựa chọn
    total += ways
    print(f"k = {k}: có {ways} số đúng {k} chữ số; cộng dồn (≤ {k} chữ số) = {total}")

k = 1: có 9 số đúng 1 chữ số; cộng dồn (≤ 1 chữ số) = 10
k = 2: có 81 số đúng 2 chữ số; cộng dồn (≤ 2 chữ số) = 91
k = 3: có 648 số đúng 3 chữ số; cộng dồn (≤ 3 chữ số) = 739
k = 4: có 4536 số đúng 4 chữ số; cộng dồn (≤ 4 chữ số) = 5275


## 5. Cách 1 — Brute Force / baseline: duyệt mọi số, kiểm tra chữ số

**Ý tưởng.** Làm đúng như đề: với mỗi `x` trong `range(10 ** n)`, đếm nếu `len(set(str(x))) == len(str(x))`.

In [9]:
def count_bruteforce(n: int) -> int:
    count = 0
    for x in range(10 ** n):
        s = str(x)
        if len(set(s)) == len(s):
            count += 1
    return count


for n, expected in examples:
    print(n, count_bruteforce(n), "(kỳ vọng", expected, ")")

2 91 (kỳ vọng 91 )
0 1 (kỳ vọng 1 )
1 10 (kỳ vọng 10 )
3 739 (kỳ vọng 739 )


**Độ phức tạp (Cách 1).** Có `10^n` số, mỗi số có tối đa `n` chữ số nên kiểm tra tốn `O(n)` ⇒ **Time `O(n · 10^n)`**; **Space `O(n)`** (chuỗi tạm). Với `n = 8` là `10^8` số — **quá chậm** trong Python (vài chục giây đến vài phút, tùy máy; vượt giới hạn thời gian của LeetCode).

**Nút thắt:** ta duyệt cả những số **không hợp lệ** (đa số). Thực tế chỉ có `2 345 851` số hợp lệ trong `10^8` số. Cần **đếm** thay vì **duyệt**.

In [10]:
# Đo thời gian vét cạn cho n = 1..6 để thấy mỗi lần tăng n, thời gian nhân ~10
prev = None
for n in range(1, 7):
    t0 = time.perf_counter()
    count_bruteforce(n)
    dt = time.perf_counter() - t0
    print(f"n = {n}: {dt:.3f} giây" + (f" (gấp {dt / prev:.1f} lần n trước)" if prev else ""))
    prev = dt
print("=> ước lượng n = 8 mất khoảng", f"{prev * 100:.0f}", "giây trên máy này — quá chậm so với giới hạn thời gian của LeetCode")

n = 1: 0.000 giây
n = 2: 0.000 giây (gấp 3.1 lần n trước)
n = 3: 0.000 giây (gấp 8.4 lần n trước)
n = 4: 0.002 giây (gấp 14.3 lần n trước)
n = 5: 0.021 giây (gấp 8.9 lần n trước)


n = 6: 0.235 giây (gấp 11.1 lần n trước)
=> ước lượng n = 8 mất khoảng 24 giây trên máy này — quá chậm so với giới hạn thời gian của LeetCode


## 6. Ý tưởng thuật toán / bước đột phá

**Đếm thay vì duyệt.** Số lượng số hợp lệ có đúng `k` chữ số **chỉ phụ thuộc `k`** và có công thức nhân:

> `g(1) = 9`, `g(k) = 9 · 9 · 8 ⋯ (11 − k)` với `k >= 2`, tức `g(k) = g(k − 1) · (11 − k)`.

Đáp án là `1 + g(1) + g(2) + ... + g(n)` (`1` cho số `0`). Với `n = 0`: chỉ có số `0` ⇒ `1`.

Vì chỉ có `10` chữ số nên `g(11) = g(10) · 0 = 0`: mọi `k > 10` cho `0` số — ràng buộc `n <= 8` còn nằm trong vùng an toàn (nếu `n` lớn hơn `10` thì đáp án không tăng thêm).

Ngoài công thức, còn các cách "máy móc" hơn nhưng tổng quát (dùng được khi đề phức tạp hơn): **quay lui** đếm từng số và **Digit DP** với `mask` các chữ số đã dùng.

## 7. Các cách tiếp theo

### Cách 2 — Quy tắc nhân, cộng dần theo độ dài (công thức)

**Giải thích từng đoạn.**

1. `if n == 0: return 1` — chỉ có số `0`.
2. Khởi tạo `ans = 10` (với `n = 1`: các số `0..9`), `cur = 9` (số cách xếp chữ số đầu của số `k = 1` chữ số), `avail = 9` (số chữ số còn lại để chọn cho vị trí kế tiếp: `10 − 1 = 9`).
3. Với mỗi `k = 2..n`: `cur *= avail` (thêm một chữ số, nhân với số lựa chọn còn lại), `ans += cur`, `avail -= 1`.
4. Trả `ans`.

Trace nhanh `n = 3`: `ans = 10, cur = 9, avail = 9` → `k = 2`: `cur = 81`, `ans = 91`, `avail = 8` → `k = 3`: `cur = 648`, `ans = 739`.

In [11]:
def count_formula(n: int) -> int:
    if n == 0:
        return 1
    ans, cur, avail = 10, 9, 9
    for _ in range(2, n + 1):
        cur *= avail
        ans += cur
        avail -= 1
    return ans


for n, expected in examples:
    print(n, count_formula(n), "(kỳ vọng", expected, ")")

2 91 (kỳ vọng 91 )
0 1 (kỳ vọng 1 )
1 10 (kỳ vọng 10 )
3 739 (kỳ vọng 739 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionFormula:
    def countNumbersWithUniqueDigits(self, n: int) -> int:
        if n == 0:
            return 1
        ans, cur, avail = 10, 9, 9
        for _ in range(2, n + 1):
            cur *= avail
            ans += cur
            avail -= 1
        return ans


print(SolutionFormula().countNumbersWithUniqueDigits(2))     # 91

91


**Độ phức tạp Cách 2:** vòng lặp `n − 1` lần ⇒ **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — Quay lui (DFS): mỗi số hợp lệ là một nút của cây "thêm chữ số"

**Ý tưởng.** Một **tiền tố** của số có chữ số khác nhau cũng có chữ số khác nhau. Vì vậy các số hợp lệ tạo thành một **cây**: gốc là `0`; con của số hiện tại là "thêm một chữ số **chưa dùng** ở bên phải". Đếm **mọi nút** của cây (độ sâu tối đa `n` chữ số) là đếm mọi số hợp lệ **đúng một lần**.

**Giải thích từng đoạn.**

1. Gốc: số `0` (`1` nút). Chữ số đầu của các số còn lại: `1..9` (không `0` để tránh `0` đứng đầu).
2. `dfs(length, mask)`: nút hiện tại là một số có `length` chữ số, `mask` là tập chữ số đã dùng. Nó đóng góp `1` (chính nó); nếu `length < n` thì duyệt `d` trong `0..9` chưa dùng, cộng `dfs(length + 1, mask | 1 << d)`.
3. Đáp án: `1 + Σ_{d=1..9} dfs(1, 1 << d)` nếu `n >= 1`; `1` nếu `n = 0`.

In [13]:
def count_dfs(n: int) -> int:
    def dfs(length, mask):
        total = 1                                    # chính số hiện tại (length chữ số)
        if length == n:
            return total
        for d in range(10):
            if not (mask >> d & 1):                  # chữ số d chưa dùng
                total += dfs(length + 1, mask | 1 << d)
        return total

    if n == 0:
        return 1                                     # chỉ có số 0
    return 1 + sum(dfs(1, 1 << d) for d in range(1, 10))     # 1: số 0; chữ số đầu 1..9


for n, expected in examples:
    print(n, count_dfs(n), "(kỳ vọng", expected, ")")


# Sinh thật các số bằng cùng cây (n nhỏ) để thấy "mỗi nút = một số hợp lệ"
def gen_numbers(n):
    out = [0]
    def dfs(value, length, mask):
        out.append(value)
        if length == n:
            return
        for d in range(10):
            if not (mask >> d & 1):
                dfs(value * 10 + d, length + 1, mask | 1 << d)
    if n >= 1:
        for d in range(1, 10):
            dfs(d, 1, 1 << d)
    return out

nums2 = gen_numbers(2)
print(len(nums2), nums2[:12], "...", nums2[-5:])
assert sorted(nums2) == [x for x in range(100) if has_unique_digits(x)]

2 91 (kỳ vọng 91 )
0 1 (kỳ vọng 1 )
1 10 (kỳ vọng 10 )
3 739 (kỳ vọng 739 )
91 [0, 1, 10, 12, 13, 14, 15, 16, 17, 18, 19, 2] ... [94, 95, 96, 97, 98]


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionDFS:
    def countNumbersWithUniqueDigits(self, n: int) -> int:
        def dfs(length: int, mask: int) -> int:
            total = 1
            if length == n:
                return total
            for d in range(10):
                if not (mask >> d & 1):
                    total += dfs(length + 1, mask | 1 << d)
            return total

        if n == 0:
            return 1
        return 1 + sum(dfs(1, 1 << d) for d in range(1, 10))


print(SolutionDFS().countNumbersWithUniqueDigits(3))     # 739

739


**Độ phức tạp Cách 3:** số nút bằng đáp án `A(n)` (với `n = 8`: `2 345 851`), mỗi nút lặp `10` chữ số ⇒ **Time `O(10 · A(n))`** (≈ vài chục triệu phép với `n = 8` — còn chậm hơn công thức rất nhiều), **Space `O(n)`** (độ sâu đệ quy).

Cách này "máy móc" nhưng **tổng quát**: khi đề thêm điều kiện lạ (ví dụ cấm chữ số `4`, hoặc phải chia hết cho `7`) thì chỉ cần thêm điều kiện vào `if` bên trong, trong khi công thức tổ hợp phải nghĩ lại từ đầu.

### Cách 4 — Digit DP `dfs(i, mask, lead)` với `@cache`

**Ý tưởng.** Đếm số theo **vị trí** từ chữ số cao nhất tới thấp nhất, cho `n` vị trí (cho phép `0` ở đầu rồi coi là "số ngắn hơn"). Trạng thái cần nhớ:

- `i`: còn lại bao nhiêu vị trí (từ `n − 1` xuống `0`; `i < 0` nghĩa là đã điền xong);
- `mask`: các chữ số đã dùng;
- `lead`: toàn bộ các vị trí đã điền **vẫn chỉ là `0` ở đầu** (chưa bắt đầu số thật) — khi đó điền `0` **không tính là dùng** chữ số `0`.

**Giải thích từng đoạn.**

1. `if i < 0: return 1` — hoàn tất một cách điền hợp lệ (đúng một số).
2. Với mỗi chữ số `j = 0..9`: nếu `mask >> j & 1` ⇒ đã dùng, bỏ. Nếu `lead and j == 0` ⇒ vẫn đang "0 đứng đầu": gọi `dfs(i − 1, mask, True)` (không đánh dấu `0`). Ngược lại: `dfs(i − 1, mask | 1 << j, False)`.
3. Tổng các nhánh là đáp án; `@cache` để các trạng thái `(i, mask, lead)` trùng nhau chỉ tính một lần.
4. Gọi `dfs(n − 1, 0, True)`. Với `n = 0`: `dfs(−1, 0, True) = 1` (số `0`).

Chuỗi toàn `0` (kể cả `lead` còn `True` đến cuối) chính là số `0` — được đếm đúng một lần.

In [15]:
def count_digit_dp(n: int) -> int:
    @cache
    def dfs(i, mask, lead):
        if i < 0:
            return 1                                  # điền xong một số hợp lệ
        ans = 0
        for j in range(10):
            if mask >> j & 1:                         # chữ số j đã dùng
                continue
            if lead and j == 0:                       # vẫn là các số 0 đứng đầu: không đánh dấu dùng
                ans += dfs(i - 1, mask, True)
            else:
                ans += dfs(i - 1, mask | 1 << j, False)
        return ans

    return dfs(n - 1, 0, True)


for n, expected in examples:
    print(n, count_digit_dp(n), "(kỳ vọng", expected, ")")

2 91 (kỳ vọng 91 )
0 1 (kỳ vọng 1 )
1 10 (kỳ vọng 10 )
3 739 (kỳ vọng 739 )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionDigitDP:
    def countNumbersWithUniqueDigits(self, n: int) -> int:
        @cache
        def dfs(i: int, mask: int, lead: bool) -> int:
            if i < 0:
                return 1
            ans = 0
            for j in range(10):
                if mask >> j & 1:
                    continue
                if lead and j == 0:
                    ans += dfs(i - 1, mask, True)
                else:
                    ans += dfs(i - 1, mask | 1 << j, False)
            return ans

        return dfs(n - 1, 0, True)


print(SolutionDigitDP().countNumbersWithUniqueDigits(8))     # 2345851

2345851


**Độ phức tạp Cách 4:** có `n` vị trí × `2^10` mask × `2` giá trị `lead` trạng thái, mỗi trạng thái lặp `10` chữ số ⇒ **Time `O(n · 10 · 2^10)`** ≈ `8 · 10 · 1024 · 2`, rất nhỏ; **Space `O(n · 2^10)`** cho bộ nhớ đệm. Đây là mẫu **Digit DP**, dùng lại cho các bài "đếm số trong `[0, N]` thỏa điều kiện về chữ số" (233, 902, 1012, 2376...).

### Cách 5 — Công thức đóng với `math.perm` / bảng đáp án

**Ý tưởng.** Số có đúng `k >= 1` chữ số khác nhau: chọn chữ số đầu `1..9` (`9` cách) rồi xếp `k − 1` chữ số từ `9` chữ số còn lại (**chỉnh hợp** `P(9, k − 1)`): `g(k) = 9 · P(9, k − 1) = 9 · perm(9, k − 1)`. Đáp án `A(n) = 1 + Σ_{k=1..n} 9 · perm(9, k − 1)`.

Vì `n` chỉ có `9` giá trị (`0..8`), ta cũng có thể **tính sẵn bảng** `A(0..8)` rồi tra `table[n]`.

In [17]:
def count_perm(n: int) -> int:
    return 1 + sum(9 * perm(9, k - 1) for k in range(1, n + 1))


TABLE = [count_perm(n) for n in range(9)]
print(TABLE)       # [1, 10, 91, 739, 5275, 32491, 168571, 712891, 2345851]


def count_table(n: int) -> int:
    return TABLE[n]


for n, expected in examples:
    print(n, count_perm(n), count_table(n), "(kỳ vọng", expected, ")")

[1, 10, 91, 739, 5275, 32491, 168571, 712891, 2345851]
2 91 91 (kỳ vọng 91 )
0 1 1 (kỳ vọng 1 )
1 10 10 (kỳ vọng 10 )
3 739 739 (kỳ vọng 739 )


In [18]:
# Code nộp LeetCode cho Cách 5 (bảng đã được tính và kiểm chứng ở trên)
class SolutionTable:
    def countNumbersWithUniqueDigits(self, n: int) -> int:
        return [1, 10, 91, 739, 5275, 32491, 168571, 712891, 2345851][n]


print(SolutionTable().countNumbersWithUniqueDigits(5))     # 32491

32491


**Độ phức tạp Cách 5:** công thức `O(n)` (mỗi `perm(9, j)` tính qua tích) hoặc tra bảng `O(1)` — **Space `O(1)`** (bảng `9` phần tử). Dùng bảng là "hợp lệ" vì ràng buộc đề rất nhỏ, nhưng **đừng học thuộc** — hãy hiểu công thức để biết vì sao các số đó.

## 8. TRACE từng bước

### 8.1. Trace công thức (Cách 2) cho `n = 4`

| `k` | `avail` trước | `cur` (số có đúng `k` chữ số) | `ans` (≤ `k` chữ số) | `avail` sau |
|---|---|---|---|---|
| 1 | — (khởi tạo) | `9` | `10` | `9` |
| 2 | `9` | `9 · 9 = 81` | `91` | `8` |
| 3 | `8` | `81 · 8 = 648` | `739` | `7` |
| 4 | `7` | `648 · 7 = 4536` | `5275` | `6` |

In [19]:
def trace_formula(n):
    print("n =", n)
    if n == 0:
        print("  chỉ có số 0 -> 1")
        return 1
    ans, cur, avail = 10, 9, 9
    print(f"  k = 1: cur = {cur}, ans = {ans}, avail = {avail}")
    for k in range(2, n + 1):
        before = avail
        cur *= avail
        ans += cur
        avail -= 1
        print(f"  k = {k}: cur = cur · {before} = {cur}, ans = {ans}, avail -> {avail}")
    return ans


assert trace_formula(4) == 5275
assert trace_formula(0) == 1

n = 4
  k = 1: cur = 9, ans = 10, avail = 9
  k = 2: cur = cur · 9 = 81, ans = 91, avail -> 8
  k = 3: cur = cur · 8 = 648, ans = 739, avail -> 7
  k = 4: cur = cur · 7 = 4536, ans = 5275, avail -> 6
n = 0
  chỉ có số 0 -> 1


### 8.2. Trace quay lui (Cách 3) cho `n = 2`: đếm theo chữ số đầu

Mỗi chữ số đầu `d = 1..9` sinh ra `1` (số `d`) `+ 9` (`d0, d1, ...` với chữ số thứ hai khác `d`) `= 10` nút. Tổng `1 + 9 · 10 = 91`.

In [20]:
def trace_dfs(n):
    total = 1
    print(f"n = {n}: gốc là số 0 -> đếm 1")
    for first in range(1, 10):
        nodes = []

        def dfs(value, length, mask):
            nodes.append(value)
            if length == n:
                return
            for d in range(10):
                if not (mask >> d & 1):
                    dfs(value * 10 + d, length + 1, mask | 1 << d)

        dfs(first, 1, 1 << first)
        total += len(nodes)
        print(f"  chữ số đầu {first}: {len(nodes)} nút: {nodes}")
    print("tổng =", total)
    return total


assert trace_dfs(2) == 91

n = 2: gốc là số 0 -> đếm 1
  chữ số đầu 1: 10 nút: [1, 10, 12, 13, 14, 15, 16, 17, 18, 19]
  chữ số đầu 2: 10 nút: [2, 20, 21, 23, 24, 25, 26, 27, 28, 29]
  chữ số đầu 3: 10 nút: [3, 30, 31, 32, 34, 35, 36, 37, 38, 39]
  chữ số đầu 4: 10 nút: [4, 40, 41, 42, 43, 45, 46, 47, 48, 49]
  chữ số đầu 5: 10 nút: [5, 50, 51, 52, 53, 54, 56, 57, 58, 59]
  chữ số đầu 6: 10 nút: [6, 60, 61, 62, 63, 64, 65, 67, 68, 69]
  chữ số đầu 7: 10 nút: [7, 70, 71, 72, 73, 74, 75, 76, 78, 79]
  chữ số đầu 8: 10 nút: [8, 80, 81, 82, 83, 84, 85, 86, 87, 89]
  chữ số đầu 9: 10 nút: [9, 90, 91, 92, 93, 94, 95, 96, 97, 98]
tổng = 91


### 8.3. Trace Digit DP: bao nhiêu trạng thái thật sự được tính?

Nhờ `@cache`, số lần gọi **thật sự tính** (`misses`) nhỏ hơn rất nhiều so với số lần gọi tổng (`hits + misses`). In `cache_info()` cho từng `n`.

In [21]:
def trace_digit_dp(n):
    @cache
    def dfs(i, mask, lead):
        if i < 0:
            return 1
        ans = 0
        for j in range(10):
            if mask >> j & 1:
                continue
            if lead and j == 0:
                ans += dfs(i - 1, mask, True)
            else:
                ans += dfs(i - 1, mask | 1 << j, False)
        return ans

    res = dfs(n - 1, 0, True)
    info = dfs.cache_info()
    print(f"n = {n}: kết quả {res}; trạng thái đã tính (misses) = {info.misses}; lần dùng lại kết quả (hits) = {info.hits}")
    return res


for n in range(0, 9):
    trace_digit_dp(n)

n = 0: kết quả 1; trạng thái đã tính (misses) = 1; lần dùng lại kết quả (hits) = 0
n = 1: kết quả 10; trạng thái đã tính (misses) = 11; lần dùng lại kết quả (hits) = 0
n = 2: kết quả 91; trạng thái đã tính (misses) = 66; lần dùng lại kết quả (hits) = 36
n = 3: kết quả 739; trạng thái đã tính (misses) = 241; lần dùng lại kết quả (hits) = 312
n = 4: kết quả 5275; trạng thái đã tính (misses) = 626; lần dùng lại kết quả (hits) = 1218
n = 5: kết quả 32491; trạng thái đã tính (misses) = 1263; lần dùng lại kết quả (hits) = 3132
n = 6: kết quả 168571; trạng thái đã tính (misses) = 2110; lần dùng lại kết quả (hits) = 6096
n = 7: kết quả 712891; trạng thái đã tính (misses) = 3077; lần dùng lại kết quả (hits) = 9780
n = 8: kết quả 2345851; trạng thái đã tính (misses) = 4089; lần dùng lại kết quả (hits) = 13779


## 9. Code nộp LeetCode

Mình nộp **Cách 2 (công thức nhân)**: `O(n)`, ngắn, không đệ quy. (Cách 4 Digit DP cũng ngắn và là mẫu tổng quát — đều đúng.)

In [22]:
class Solution:
    def countNumbersWithUniqueDigits(self, n: int) -> int:
        if n == 0:
            return 1
        ans, cur, avail = 10, 9, 9       # ans: đã đếm (<= k chữ số); cur: số có đúng k chữ số; avail: lựa chọn còn lại
        for _ in range(2, n + 1):
            cur *= avail                  # thêm một chữ số: nhân với số lựa chọn còn lại
            ans += cur
            avail -= 1
        return ans


sol = Solution()
for n, expected in examples:
    print(n, "->", sol.countNumbersWithUniqueDigits(n), "(kỳ vọng", expected, ")")

2 -> 91 (kỳ vọng 91 )
0 -> 1 (kỳ vọng 1 )
1 -> 10 (kỳ vọng 10 )
3 -> 739 (kỳ vọng 739 )


## 10. Vì sao thuật toán đúng

- **Phân hoạch theo độ dài:** mỗi số `x` với `0 <= x < 10^n` thuộc đúng một nhóm: `x = 0`, hoặc `x` có đúng `k` chữ số với `k = 1..n` (không tính số `0` đứng đầu). Các nhóm rời nhau nên cộng số lượng từng nhóm là đúng.
- **Đếm nhóm `k` chữ số hợp lệ:** xếp các chữ số từ trái sang phải. Chữ số đầu: `1..9` ⇒ `9` cách (không `0` vì `x` có đúng `k` chữ số). Chữ số thứ `2`: `0..9` trừ chữ số đầu ⇒ `9` cách. Chữ số thứ `3`: `0..9` trừ hai chữ số đã dùng ⇒ `8` cách. Tổng quát chữ số thứ `j >= 2` có `10 − (j − 1) = 11 − j` cách **bất kể** các chữ số đã chọn cụ thể là gì (chỉ cần chúng khác nhau) ⇒ quy tắc nhân áp dụng: `g(k) = 9 · 9 · 8 ⋯ (11 − k)`.
- **Vòng lặp code khớp công thức:** `cur` ở lần lặp `k` được nhân với `avail`; `avail` bắt đầu `9` ở `k = 2` và giảm `1` mỗi lần ⇒ `avail = 11 − k`. Khớp.
- **`n = 0`:** khoảng `[0, 1)` chỉ có `0` ⇒ `1` (trường hợp riêng).
- **Không vượt `10` chữ số:** với `n <= 8` mọi `avail >= 3 > 0` nên không có nhân với `0` hay số âm.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt mọi số | `O(n · 10^n)` | `O(n)` |
| 2. Quy tắc nhân (cộng dần) | `O(n)` | `O(1)` |
| 3. Quay lui đếm từng số | `O(10 · A(n))`, `A(8) = 2 345 851` | `O(n)` |
| 4. Digit DP `(i, mask, lead)` | `O(n · 10 · 2^10)` | `O(n · 2^10)` |
| 5. `math.perm` / bảng | `O(n)` / `O(1)` | `O(1)` |

Với `n <= 8`, Cách 2/4/5 chạy tức thì; Cách 3 mất vài giây; Cách 1 không dùng được ở `n = 8`. Khi phải đếm "các số trong `[0, N]` thỏa điều kiện chữ số" cho `N` rất lớn (`10^18`), Digit DP (Cách 4 mở rộng thêm cờ `tight`) là công cụ chính — xem bài tập 5.

## 12. Bộ test

Với mỗi `n` từ `0` đến `8`, ta so sánh các cách với **hai oracle độc lập**:

- vét cạn `count_bruteforce` (chỉ dùng được tới `n <= 6` cho nhanh);
- **đếm hoán vị** `count_by_permutations(n)` dùng `itertools.permutations` (khác hẳn công thức): `1 + Σ_k #{bộ k chữ số khác nhau, chữ số đầu ≠ 0}` — chạy được tới `n = 8`.

In [23]:
def count_by_permutations(n):
    total = 1                                            # số 0
    for k in range(1, n + 1):
        total += sum(1 for p in permutations(range(10), k) if p[0] != 0)
    return total


EXPECTED = [1, 10, 91, 739, 5275, 32491, 168571, 712891, 2345851]

for n in range(9):
    got_perm = count_by_permutations(n)
    assert got_perm == EXPECTED[n], (n, got_perm)
    if n <= 6:
        assert count_bruteforce(n) == EXPECTED[n], n
    assert count_formula(n) == EXPECTED[n]
    assert count_digit_dp(n) == EXPECTED[n]
    assert count_perm(n) == EXPECTED[n]
    assert count_table(n) == EXPECTED[n]
    assert Solution().countNumbersWithUniqueDigits(n) == EXPECTED[n]
    assert SolutionFormula().countNumbersWithUniqueDigits(n) == EXPECTED[n]
    assert SolutionDigitDP().countNumbersWithUniqueDigits(n) == EXPECTED[n]
    assert SolutionTable().countNumbersWithUniqueDigits(n) == EXPECTED[n]
    print(f"n = {n}: {EXPECTED[n]}")
print("Công thức, Digit DP, perm, bảng, Solution khớp oracle hoán vị với mọi n = 0..8")

n = 0: 1
n = 1: 10
n = 2: 91
n = 3: 739
n = 4: 5275
n = 5: 32491


n = 6: 168571
n = 7: 712891
n = 8: 2345851
Công thức, Digit DP, perm, bảng, Solution khớp oracle hoán vị với mọi n = 0..8


In [24]:
# Quay lui (chậm hơn) chỉ chạy cho n <= 6, rồi n = 7, 8 một lần để thấy vẫn đúng
t0 = time.perf_counter()
for n in range(0, 9):
    assert count_dfs(n) == EXPECTED[n], n
    assert SolutionDFS().countNumbersWithUniqueDigits(n) == EXPECTED[n]
print(f"quay lui khớp với mọi n = 0..8 ({time.perf_counter() - t0:.1f} giây)")

quay lui khớp với mọi n = 0..8 (1.1 giây)


### 12.2. Biên, đơn điệu và tính chất

- `n = 0` ⇒ `1`; `n = 1` ⇒ `10`.
- Đáp án **tăng** theo `n` (mỗi nhóm `g(k) > 0` với `k <= 10`).
- Hiệu `A(n) − A(n − 1) = g(n)` = số có đúng `n` chữ số hợp lệ.
- `A(n) <= 10^n` (không thể nhiều hơn tổng số).
- Các số hợp lệ có đúng `k` chữ số khớp với vét cạn trên khoảng `[10^(k−1), 10^k)`.

In [25]:
for n in range(1, 9):
    assert EXPECTED[n] > EXPECTED[n - 1]
    assert EXPECTED[n] <= 10 ** n
    g = EXPECTED[n] - EXPECTED[n - 1]
    assert g == 9 * perm(9, n - 1)
    if n <= 6:
        lo, hi = (0 if n == 1 else 10 ** (n - 1)), 10 ** n
        assert g == sum(1 for x in range(max(lo, 1), hi) if has_unique_digits(x)), n
print("tính chất đơn điệu và chênh lệch theo từng độ dài đều đúng")

tính chất đơn điệu và chênh lệch theo từng độ dài đều đúng


### 12.3. Phản ví dụ cho các cách làm sai

Bốn hàm **sai** dưới đây (mỗi hàm là một lỗi ở mục 13); in ra để thấy chúng lệch với `n = 0, 1, 2, 3`.

In [26]:
def bad_no_zero_case(n):
    """SAI: quên n = 0 (chạy công thức thẳng)."""
    ans, cur, avail = 10, 9, 9
    for _ in range(2, n + 1):
        cur *= avail
        ans += cur
        avail -= 1
    return ans


def bad_first_digit_10(n):
    """SAI: chữ số đầu cũng có 10 cách (tính cả 0 đứng đầu)."""
    ans = 1
    for k in range(1, n + 1):
        ways = 10
        for j in range(1, k):
            ways *= 10 - j
        ans += ways
    return ans


def bad_only_exact_n(n):
    """SAI: chỉ đếm số có đúng n chữ số."""
    if n == 0:
        return 1
    cur = 9
    for j in range(1, n):
        cur *= 10 - j
    return cur


def bad_no_lead(n):
    """SAI: Digit DP bỏ cờ lead (0 đứng đầu bị tính là đã dùng chữ số 0)."""
    @cache
    def dfs(i, mask):
        if i < 0:
            return 1
        return sum(dfs(i - 1, mask | 1 << j) for j in range(10) if not mask >> j & 1)
    return dfs(n - 1, 0)


print("n          :", list(range(5)))
print("đúng       :", EXPECTED[:5])
for f in (bad_no_zero_case, bad_first_digit_10, bad_only_exact_n, bad_no_lead):
    print(f"{f.__name__:18s}:", [f(n) for n in range(5)])
assert bad_no_zero_case(0) != 1
assert bad_first_digit_10(2) != 91
assert bad_only_exact_n(2) != 91
assert bad_no_lead(2) != 91

n          : [0, 1, 2, 3, 4]
đúng       : [1, 10, 91, 739, 5275]
bad_no_zero_case  : [10, 10, 91, 739, 5275]
bad_first_digit_10: [1, 11, 101, 821, 5861]
bad_only_exact_n  : [1, 9, 81, 648, 4536]
bad_no_lead       : [1, 10, 90, 720, 5040]


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên trường hợp `n = 0`

**Code sai.** Chạy thẳng công thức khởi tạo `ans = 10`:
```python
ans, cur, avail = 10, 9, 9
for _ in range(2, n + 1): ...
return ans            # n = 0 vẫn trả 10, đúng phải là 1
```
**Vì sao sai.** `ans = 10` ứng với `n = 1` (các số `0..9`). `n = 0` là khoảng `[0, 1)` chỉ có số `0`.

**Code đúng.** `if n == 0: return 1` đầu hàm.

In [27]:
print("sai :", bad_no_zero_case(0), "(phải là 1)")
print("đúng:", Solution().countNumbersWithUniqueDigits(0))

sai : 10 (phải là 1)
đúng: 1


### Lỗi 2: Chữ số đầu có `10` cách (tính cả `0` đứng đầu)

**Code sai.** `10 · 9 · 8 ⋯` cho `k` chữ số.

**Vì sao sai.** Chuỗi `"05"` không phải số có `2` chữ số, nó chính là số `5` (đã được đếm ở nhóm `1` chữ số) ⇒ **đếm trùng**. Với `n = 2`: `10 + 90 + 1 = 101`, không phải `91`.

**Code đúng.** Chữ số đầu `9` cách (`1..9`), các chữ số sau `9, 8, ...`.

In [28]:
print("sai :", [bad_first_digit_10(n) for n in range(4)])      # [1, 11, 101, ...]
print("đúng:", EXPECTED[:4])

sai : [1, 11, 101, 821]
đúng: [1, 10, 91, 739]


### Lỗi 3: Chỉ đếm số có **đúng** `n` chữ số

**Code sai.** Trả `9 · 9 · 8 ⋯` cho `k = n` mà không cộng các nhóm ngắn hơn.

**Vì sao sai.** Khoảng `0 <= x < 10^n` gồm cả số có ít chữ số hơn (`x = 5`, `x = 73`, ...). Với `n = 2`, `81` chỉ là số hai chữ số; đáp án là `91`.

**Code đúng.** Cộng dồn `g(1) + g(2) + ... + g(n)` và số `0`.

In [29]:
print("sai :", [bad_only_exact_n(n) for n in range(4)])      # [1, 9, 81, 648]
print("đúng:", EXPECTED[:4])

sai : [1, 9, 81, 648]
đúng: [1, 10, 91, 739]


### Lỗi 4: Digit DP bỏ cờ `lead`

**Code sai.** `dfs(i, mask)` luôn đánh dấu chữ số vừa chọn là đã dùng, kể cả `0` ở đầu.

**Vì sao sai.** Các `0` ở đầu không phải chữ số thật: ta đang "đệm" số ngắn hơn `n` chữ số bằng các `0` ở đầu. Nếu những `0` đệm đó bị coi là đã dùng chữ số `0`, thì các số ngắn hơn mà vốn **chứa** chữ số `0` hoặc cần nhiều `0` đệm sẽ bị loại nhầm (với `n = 3`: số `5` thành `"005"` và số `10` thành `"010"`, đều có hai chữ số `0` trùng nhau). Với `n = 2`, số `0` (`"00"`) bị loại nên kết quả là `90`, không phải `91`.

**Code đúng.** Giữ cờ `lead`: khi `lead` còn `True` và chọn `0` thì **không** cập nhật `mask`.

In [30]:
print("sai :", [bad_no_lead(n) for n in range(5)])
print("đúng:", EXPECTED[:5])

sai : [1, 10, 90, 720, 5040]
đúng: [1, 10, 91, 739, 5275]


### Lỗi 5: Vét cạn `10^8` số khi `n = 8`

**Code sai.** Cách 1 (`for x in range(10 ** n)`).

**Vì sao sai.** `10^8` lần `str` + `set` trong Python mất từ vài chục giây đến vài phút — **Time Limit Exceeded**. Thời gian đo ở mục 5 tăng gấp `~10` mỗi lần `n` tăng `1`.

**Code đúng.** Đếm bằng công thức (Cách 2) hoặc Digit DP (Cách 4).

In [31]:
t0 = time.perf_counter()
Solution().countNumbersWithUniqueDigits(8)
print(f"công thức n = 8 chỉ mất {time.perf_counter() - t0:.6f} giây")

công thức n = 8 chỉ mất 0.000053 giây


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số có chữ số khác nhau" ⇒ **quy tắc nhân**: chọn từng vị trí, số lựa chọn giảm dần `9, 9, 8, 7, ...` (chữ số đầu không được `0`).
- Chia bài theo **độ dài** `k = 1..n` rồi cộng; nhớ số `0` riêng.
- "Đếm số trong khoảng `[L, R]` theo điều kiện chữ số" ⇒ **Digit DP**: `dfs(vị trí, trạng thái, cờ lead[, cờ tight])`.
- "Đã dùng chữ số nào" ⇒ **bitmask** `10` bit (`mask >> d & 1`, `mask | 1 << d`).
- Có thể **bảng hóa** khi miền vào quá nhỏ (ở đây `n ∈ [0, 8]`): nhưng hãy chứng minh bằng code rồi mới hard-code.
- Đáp án không thay đổi với `n > 10` (không có số `11` chữ số khác nhau): trong công thức `avail` về `0` làm `cur = 0`.
- Cách kiểm tra độc lập hay dùng: **sinh hoán vị** (`itertools.permutations`) hoặc vét cạn với `n` nhỏ.
- Hiện tượng hay gặp: "đếm" rất lớn nhưng công thức nhỏ — đừng duyệt khi có thể đếm.

## 15. Biến thể của bài

1. **`n` lớn hơn:** `n = 9, 10, 11, ...`: đáp án đạt tối đa tại `n = 10` rồi giữ nguyên (`8 877 691`) — xem code bên dưới.
2. **Đếm trong `[0, N]` tùy ý** (không chỉ `10^n − 1`): Digit DP thêm cờ `tight` (bài tập 5). Đây là **LeetCode 2376 Count Special Integers**.
3. **Ngược lại:** đếm số có **ít nhất một chữ số lặp** trong `[1, N]`: `N − (số hợp lệ trong [1, N])` — **LeetCode 1012 Numbers With Repeated Digits**.
4. **Chữ số bị cấm / giới hạn tập chữ số:** LeetCode 902 (Numbers At Most N Given Digit Set); thay `range(10)` bằng tập cho trước.
5. **Đếm số có chữ số `1`:** LeetCode 233 (Number of Digit One); **Không có `1` liên tiếp (dạng nhị phân):** LeetCode 600.
6. **Hệ cơ số khác:** cơ số `b`: chữ số đầu `b − 1` cách, rồi `b − 1`, `b − 2`, ... — thử viết hàm tổng quát.

In [32]:
def count_formula_base(n, base=10):
    """Số trong [0, base^n) có các chữ số (cơ số base) đôi một khác nhau."""
    if n == 0:
        return 1
    ans, cur, avail = base, base - 1, base - 1
    for _ in range(2, n + 1):
        cur *= max(avail, 0)
        ans += cur
        avail -= 1
    return ans


print([count_formula_base(n) for n in range(0, 13)])        # đạt 8877691 tại n = 10, giữ nguyên từ đó
assert count_formula_base(10) == 8877691
assert all(count_formula_base(n) == 8877691 for n in range(10, 14))
assert [count_formula_base(n) for n in range(9)] == EXPECTED
# cơ số 2: các số nhị phân có chữ số khác nhau: 0, 1, 10 (=2) -> n = 2: 0,1,2 (11 bị loại) -> 3
assert count_formula_base(2, base=2) == 3
assert count_formula_base(3, base=2) == 3

[1, 10, 91, 739, 5275, 32491, 168571, 712891, 2345851, 5611771, 8877691, 8877691, 8877691]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt mọi số | `len(set(str(x)))` | `O(n · 10^n)` | `O(n)` | hiểu đề, oracle cho `n` nhỏ |
| 2. Quy tắc nhân | `9 · 9 · 8 ⋯`, cộng theo độ dài | `O(n)` | `O(1)` | **nên nộp** |
| 3. Quay lui | mỗi số hợp lệ = một nút của cây | `O(10 · A(n))` | `O(n)` | khi thêm điều kiện lạ lên từng chữ số |
| 4. Digit DP | `dfs(i, mask, lead)` + cache | `O(n · 10 · 2^10)` | `O(n · 2^10)` | mẫu tổng quát cho bài đếm số theo chữ số |
| 5. `math.perm` / bảng | `1 + Σ 9 · perm(9, k − 1)` | `O(n)` / `O(1)` | `O(1)` | ngắn gọn; đã chứng minh công thức |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `9 * 9 * 8 * 7`, `len(set(str(1223)))`, `has_unique_digits(907)`, `Solution().countNumbersWithUniqueDigits(1)`.

**Bài 2 (tính tay).** Tính `countNumbersWithUniqueDigits(4)` bằng bảng (từng nhóm `k = 1, 2, 3, 4` rồi cộng số `0`). Đáp án phải là `5275`.

**Bài 3 (điền TODO).** Hoàn thành `count_formula_todo(n)` theo Cách 2.

In [33]:
def count_formula_todo(n):
    # TODO 1: if n == 0: return 1
    # TODO 2: ans, cur, avail = 10, 9, 9
    # TODO 3: for _ in range(2, n + 1): cur *= avail; ans += cur; avail -= 1
    # TODO 4: return ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết lại **từ đầu** Cách 4 (Digit DP) `count_dp_mine(n)` — nhớ ba tham số `(i, mask, lead)` và điều kiện dừng `i < 0`.

**Bài 5 (biến thể / mini challenge).** Viết `count_unique_upto(N)`: đếm các số `x` với `0 <= x <= N` có chữ số khác nhau, với `N` **bất kỳ** (ví dụ `N = 5000`, `N = 10**18`). Gợi ý: Digit DP trên các chữ số của `N`, thêm cờ `tight` ("các chữ số đã điền còn bằng tiền tố của `N`"). Kiểm tra bằng vét cạn với `N <= 5000`. Sau đó viết `count_repeated_upto(N) = (N + 1) - count_unique_upto(N)` (số có chữ số lặp trong `[0, N]`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `9 · 9 · 8 · 7 = 4536`; `str(1223) = "1223"`, `set = {"1", "2", "3"}` ⇒ `3`; `907` các chữ số `9, 0, 7` khác nhau ⇒ `True`; `n = 1` ⇒ `10`. Bài 3: bốn dòng đúng theo bốn TODO.
- **Mức 2 (vừa):** Bài 2: `g(1) = 9`, `g(2) = 81`, `g(3) = 648`, `g(4) = 4536`; cộng `1` (số `0`): `1 + 9 + 81 + 648 + 4536 = 5275`. Bài 4: `for j in range(10): if mask >> j & 1: continue; if lead and j == 0: ... else: ...`.
- **Mức 3 (sâu):** Bài 5: `dfs(i, mask, tight, lead)`; chữ số tối đa tại vị trí `i` là `digits[i]` nếu `tight` else `9`; tiếp tục `tight` mới = `tight and d == digits[i]`; khi `lead` và `d == 0` thì giữ `mask`; `i == len(digits)` ⇒ `1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1
assert 9 * 9 * 8 * 7 == 4536
assert len(set(str(1223))) == 3
assert has_unique_digits(907) is True
assert Solution().countNumbersWithUniqueDigits(1) == 10

# Bài 2
assert 1 + 9 + 81 + 648 + 4536 == 5275 == Solution().countNumbersWithUniqueDigits(4)


# Bài 3
def count_formula_todo_ans(n):
    if n == 0:
        return 1
    ans, cur, avail = 10, 9, 9
    for _ in range(2, n + 1):
        cur *= avail
        ans += cur
        avail -= 1
    return ans


assert [count_formula_todo_ans(n) for n in range(9)] == EXPECTED


# Bài 4
def count_dp_mine(n):
    @cache
    def dfs(i, mask, lead):
        if i < 0:
            return 1
        ans = 0
        for j in range(10):
            if mask >> j & 1:
                continue
            if lead and j == 0:
                ans += dfs(i - 1, mask, True)
            else:
                ans += dfs(i - 1, mask | 1 << j, False)
        return ans
    return dfs(n - 1, 0, True)


assert [count_dp_mine(n) for n in range(9)] == EXPECTED


# Bài 5
def count_unique_upto(N):
    digits = list(map(int, str(N)))
    L = len(digits)

    @cache
    def dfs(i, mask, tight, lead):
        if i == L:
            return 1                                  # số 0 (lead đến cuối) cũng được đếm 1 lần
        hi = digits[i] if tight else 9
        total = 0
        for d in range(hi + 1):
            if lead and d == 0:
                total += dfs(i + 1, mask, tight and d == hi, True)
            elif not (mask >> d & 1):
                total += dfs(i + 1, mask | 1 << d, tight and d == hi, False)
        return total

    return dfs(0, 0, True, True)


def count_repeated_upto(N):
    return (N + 1) - count_unique_upto(N)


for N in list(range(0, 400)) + [999, 1000, 4999, 5000]:
    brute = sum(1 for x in range(N + 1) if has_unique_digits(x))
    assert count_unique_upto(N) == brute, N
    assert count_repeated_upto(N) == N + 1 - brute
assert count_unique_upto(10 ** 2 - 1) == 91
assert count_unique_upto(10 ** 8 - 1) == 2345851
assert count_repeated_upto(100) == 10                   # 11, 22, ..., 99 và 100 -> 10 số lặp trong [0, 100]
print("count_unique_upto(10**18) =", count_unique_upto(10 ** 18))
print("Tất cả đáp án bài tập đều đúng")

count_unique_upto(10**18) = 8877691
Tất cả đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (quy tắc nhân)

```python
class Solution:
    def countNumbersWithUniqueDigits(self, n: int) -> int:
        if n == 0:
            return 1
        ans, cur, avail = 10, 9, 9
        for _ in range(2, n + 1):
            cur *= avail          # thêm 1 chữ số: nhân với số lựa chọn còn lại
            ans += cur
            avail -= 1
        return ans
```

**Template Digit DP (mẫu tổng quát)**

```python
@cache
def dfs(i, mask, lead):
    if i < 0:
        return 1
    ans = 0
    for j in range(10):
        if mask >> j & 1:
            continue
        ans += dfs(i - 1, mask, True) if lead and j == 0 else dfs(i - 1, mask | 1 << j, False)
    return ans
# đáp án: dfs(n - 1, 0, True)
```

**Checklist**

- [ ] Đã xử lý `n = 0 ⇒ 1`?
- [ ] Chữ số đầu có `9` cách (không `0`); các chữ số sau `9, 8, 7, ...`?
- [ ] Cộng **mọi độ dài** `k = 1..n` (không chỉ đúng `n` chữ số) và số `0`?
- [ ] Nếu dùng Digit DP: có cờ `lead` (không đánh dấu `0` đứng đầu) và `@cache`?
- [ ] Không vét cạn `10^n` khi `n = 8`?
- [ ] Đã thử `n = 0, 1, 2, 3` (`1, 10, 91, 739`) và `n = 8` (`2345851`)?
- [ ] Skeleton đúng `class Solution` / `def countNumbersWithUniqueDigits(self, n: int) -> int`?